**Record of a July 2026 Colab run.** This notebook keeps the original outputs, though its first cell was replaced to clone this repository, so those outputs came from the code of the time. To reproduce the result today, use [`docs/reproducing.md`](../docs/reproducing.md).

# Phase 5 — controlled curriculum pilot

This historical pilot performs 60 FP32 optimizer steps over 15 four-completion generation groups, reloads the adapter, and evaluates it with the original strict decoder. I later found that `click-test` is seed-invariant and that this decoder differs from training, so its 20 nominal evaluation seeds are not a held-out task suite. Use `phase5_gate_a_rescore.ipynb` to interpret the saved checkpoint.

## 1. Fetch the repository

In [ ]:
# Fetch the public repository.
import os
import subprocess
from pathlib import Path

repository_path = Path('/content/browser-control')
if not repository_path.exists():
    subprocess.run(['git', 'clone', '--branch', 'lfm-browser-control',
                    'https://github.com/DerrickKirimi/browser-control.git', str(repository_path)], check=True)
os.chdir(repository_path)


## 2. Install without replacing Colab's PyTorch

In [2]:
from importlib.metadata import version

torch_before = version('torch')
%pip install -q -e ".[environment,model,training]" pytest
torch_after = version('torch')
print('PyTorch:', torch_before, '->', torch_after)
if torch_after != torch_before:
    raise RuntimeError('Project installation replaced Colab PyTorch')
%pip uninstall -y torchao
!python -m playwright install --with-deps chromium

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 136.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Validate GPU and pinned TRL semantics

In [3]:
!python scripts/check_colab_gpu.py
!python -m pytest -q tests/test_trl_contract.py

{
  "cudnn_version": 91900,
  "device": {
    "compute_capability": "7.5",
    "index": 0,
    "name": "Tesla T4",
    "supports_bfloat16": false,
    "total_memory_bytes": 15637086208,
    "training_dtype": "float16"
  },
  "torch_cuda_version": "12.8",
  "torch_version": "2.11.0+cu128"
}
...                                                                      [100%]


## 4. Run the predeclared seed-23 pilot

In [4]:
!python scripts/train_controlled.py --seed 23 --max-steps 60 --curriculum-steps 40 --output-dir artifacts/phase5/seed-23

2026-07-21 19:40:16.585071: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 91.6kB [00:00, 200MB/s]
tokenizer.json: 4.73MB [00:00, 149MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 2.80MB/s]
chat_template.jinja: 1.38kB [00:00, 6.75MB/s]
config.json: 100% 999/999 [00:00<00:00, 10.1MB/s]
model.safetensors: 100% 709M/709M [00:08<00:00, 86.0MB/s]
generation_config.json: 100% 137/137 [00:00<00:00, 927kB/s]
  0% 0/60 [00:00<?, ?it/s]Could not estimate the number of tokens of the input, floating-point operations will not be computed
{'loss': 0.6836, 'grad_norm': 5.3192338943481445, 'learning_rate': 0.0001, 'num_tokens': 623.0, 'completions/mean_length': 18.75, 'completions/min_length': 10.0, 'completions/max_length': 32

## 5. Download the controlled report and adapter

In [5]:
import json
import shutil
from pathlib import Path

from google.colab import files

output_path = Path('artifacts/phase5/seed-23')
status_path = output_path / 'run-status.json'
run_stem = 'phase5-click-test-seed-0023-steps-0060'
report_path = output_path / f'{run_stem}-report.json'
if not status_path.is_file():
    raise RuntimeError('Controlled run has no status file')
status = json.loads(status_path.read_text())
if status.get('status') != 'success':
    if report_path.is_file():
        files.download(str(report_path))
    raise RuntimeError(f'Controlled run did not succeed: {status}')
adapter_path = output_path / 'adapter'
archive = shutil.make_archive(f'{run_stem}-adapter', 'zip', adapter_path)
files.download(str(report_path))
files.download(archive)

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]